In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from vizopt import jaxopt, components
from vizopt.base import ObjectiveTerm

In [ ]:
random_labels = ["apple", "double espresso", "quiet room", "blue", "green", "yellow", "short", "extremely long"]
n_labels = len(random_labels)
rng = np.random.default_rng(0)
point_positions = 10*rng.random([n_labels, 2])
label_df = pd.DataFrame({"label": random_labels, "x": point_positions[:, 0], "y": point_positions[:, 1]})
label_df["dy"] = 1.
label_df["dx"] = label_df["label"].str.len()*0.5
label_df

In [ ]:
input_parameters = {
    "point_positions": point_positions,
    "rectangle_sizes": label_df[["dx", "dy"]].values,
}
input_parameters

In [ ]:
optim_vars = {"rectangle_positions": point_positions}
optim_vars

In [ ]:
def plot_rectangles(optim_vars, input_parameters):
    _, ax = plt.subplots(figsize=(4, 3))
    n_rect = optim_vars["rectangle_positions"].shape[0]
    for i_rect in range(n_rect):
        ax.add_patch(plt.Rectangle(
            optim_vars["rectangle_positions"][i_rect, :],
            input_parameters["rectangle_sizes"][i_rect, 0],
            input_parameters["rectangle_sizes"][i_rect, 1],
            alpha=0.5
        ))
        link_x = [optim_vars["rectangle_positions"][i_rect, 0], input_parameters["point_positions"][i_rect, 0]]
        link_y = [optim_vars["rectangle_positions"][i_rect, 1], input_parameters["point_positions"][i_rect, 1]]
        ax.plot(link_x, link_y)
        
    ax.scatter(input_parameters["point_positions"][:, 0], input_parameters["point_positions"][:, 1])

plot_rectangles(optim_vars, input_parameters)

In [ ]:
from jax import numpy as jnp


# pairwise intersections between label bounding boxes
def calculate_intersection_loss(optim_vars, input_parameters):
    bbox_matrix = jnp.stack([
        optim_vars["rectangle_positions"],
        optim_vars["rectangle_positions"] + input_parameters["rectangle_sizes"]
    ], axis=1)

    interlabel_inters_matrix = components.multiple_bbox_intersections(bbox_matrix, bbox_matrix)
    interlabel_inters_array = interlabel_inters_matrix[np.triu_indices(interlabel_inters_matrix.shape[0], 1)]
    return jnp.sum(interlabel_inters_array)

calculate_intersection_loss(optim_vars, input_parameters)

In [ ]:
# distances between points and the respective labels
def calculate_point_label_distance_loss(optim_vars, input_parameters):
    return jnp.sum(
        (optim_vars["rectangle_positions"] - input_parameters["point_positions"]) ** 2
    )


objective_terms = [
    ObjectiveTerm(name="intersection_loss", compute=calculate_intersection_loss, multiplier=5.),
    ObjectiveTerm(
        name="point_label_distance", compute=calculate_point_label_distance_loss
    ),
]

In [ ]:
def fun_to_minimize(optim_vars):
    loss = 0.0
    for obj_term in objective_terms:
        loss += obj_term.compute(optim_vars, input_parameters) * obj_term.multiplier
    return loss

optim_vars_opt, _ = jaxopt.optimize_gradient_descent(optim_vars, fun_to_minimize=fun_to_minimize, n_iters=2000)
optim_vars_opt

In [ ]:
plot_rectangles(optim_vars_opt, input_parameters)